In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys

sys.path.insert(0, "../")

from data.features import (
    agg_crops,
    agg_surplus,
    agg_weather,
    agg_weather_w_lag,
    daily_nitrate,
    lagged_sensor_nitrate,
    nitrate_rolling,
    nitrate_avg_seasonal,
    nitrate_avg_calendar,
    doy_climatology_pure_signal,
)
from data.transforms import flatten_buckets, merge_on_date, match_seasonal
from data import get_site_ids

## Question: Is lagged weather data helpful?

In [ ]:
from cook import *
from recipes2 import _covariates


def recipe_lagger(lags=[]):
    def recipe(site_uid):
        def lagged(lag):
            wdf = (agg_weather(site_uid, edges=[])
                   .sort_values("date").set_index("date").asfreq("D")  # regular daily index
                   .shift(lag))                                        # actually lag by `lag` days
            wdf.columns = [f"{c}_lag{lag}" for c in wdf.columns]       # suffix the value columns
            return wdf.reset_index()
        n_daily, parts = _covariates(site_uid)
        parts += [lagged(i) for i in lags]
        out = merge_on_date([n_daily, *parts], spine=n_daily.index)
        return out.dropna(subset=["nitrate_con"]).reset_index(drop=True)
    return recipe

recipe = recipe_lagger([1])

lags = [1, 2, 3, 7, 10, 14, 21, 30]
recipes = {f"Lags {lags[:i]}" : recipe_lagger(lags[:i]) for i in range(len(lags))}
print(compare_many(recipes, **FAST_XGB))
    

In [ ]:
from cook import *
from recipes2 import _covariates
from data.features import agg_weather, nitrate_violations   # names the recipe needs
from data.transforms import merge_on_date                   # not re-exported by `from cook import *`

def recipe_lagger(lags=[]):
    def recipe(site_uid):
        def lagged(lag):
            wdf = (agg_weather(site_uid, edges=[])
                   .sort_values("date").set_index("date").asfreq("D")
                   .shift(lag))
            wdf.columns = [f"{c}_lag{lag}" for c in wdf.columns]
            return wdf.reset_index()
        n_daily, parts = _covariates(site_uid)
        parts += [lagged(i) for i in lags]
        v = nitrate_violations(site_uid, threshold=10).rename("violation")
        out = merge_on_date([v, *parts], spine=n_daily.index)
        return out.dropna(subset=["violation"]).reset_index(drop=True)
    return recipe

lags = [1, 2, 3, 7, 10, 14, 21, 30]
recipes = {f"Lags {lags[:i]}": recipe_lagger(lags[:i]) for i in range(len(lags))}
print(compare_many(recipes, target="violation", task="clf", **FAST_XGB))   # both args set


In [ ]:
import pandas as pd
pd.read_csv("test3.csv")

In [ ]:
import sys

sys.path.insert(0, "../")

from data.features import (
    agg_crops,
    agg_surplus,
    agg_weather,
    agg_weather_w_lag,
    daily_nitrate,
    lagged_sensor_nitrate,
    nitrate_rolling,
    nitrate_avg_seasonal,
    nitrate_avg_calendar,
    doy_climatology_pure_signal,
)
from data.transforms import flatten_buckets, merge_on_date, match_seasonal
from data import get_site_ids

from cook import *
from recipes2 import _covariates


def recipe_lagger(lags=[]):
    def recipe(site_uid):
        def lagged(lag):
            wdf = (
                agg_weather(site_uid, edges=[])
                .sort_values("date")
                .set_index("date")
                .asfreq("D")  # regular daily index
                .shift(lag)
            )  # actually lag by `lag` days
            wdf.columns = [f"{c}_lag{lag}" for c in wdf.columns]  # suffix the value columns
            return wdf.reset_index()

        n_daily, parts = _covariates(site_uid)
        parts += [lagged(i) for i in lags]
        out = merge_on_date([n_daily, *parts], spine=n_daily.index)
        return out.dropna(subset=["nitrate_con"]).reset_index(drop=True)

    return recipe


recipes = {"test": recipe_lagger([1])}
sites = ["WQS0115", "WQS0039", "WQS0003"]

# lags = [1, 2, 3, 7, 10, 14, 21, 30]
# recipes = {f"Lags {lags[:i]}": recipe_lagger(lags[:i]) for i in range(len(lags))}
test = compare_many(
    recipes,
    sites=sites,
    **FAST_XGB,
)

In [ ]:
importance = pd.read_csv("test_results/experiment_weather_lag_importance.csv")
print(importance)